# ___Phylogenetic traversal of ACEs___
---------------------------------------------------

In [1]:
print(R.version$version.string)

[1] "R version 4.5.2 (2025-10-31 ucrt)"


In [2]:
suppressPackageStartupMessages({
    library("ape")
    library("phytools")
    library("corHMM")
    library("diversitree")
})

set.seed(2026-1-26)

In [3]:
STATES <- read.csv("../../data/chapter2/FREDv3subset/finalized_states_395_species.csv", stringsAsFactors = TRUE)[, c("binominal", "state")] # finalized mycorrhizal states
COLLAB_AXIS <- read.csv("../../data/chapter2/FREDv3subset/collab_ord1_species_avgs_SRL_RD.csv", stringsAsFactors = TRUE) # first order species averaged RD and SRL values
MERGED <- merge(x = STATES, y = COLLAB_AXIS, by = "binominal") # merge the two datasets into one, based on the binominal names
stopifnot(nrow(MERGED)==395)

PHYLOGENY <- ape::multi2di(ape::read.tree("../../data/chapter2/uphylomaker/FRED_subset_collab_395sp.tre")) # phylogenetic tree created for the 395 species using U.PhyloMaker
stopifnot(length(PHYLOGENY$tip.label)==395)

# MERGED contains spaces in the binominal names - replace that with underscores; and '/' in mycorrhizal states that need to be removed
data <- data.frame(binominal = gsub(MERGED$binominal, pattern = ' ', replacement = '_'), RD = MERGED$F00679, SRL = MERGED$F00727, myco = gsub(x = MERGED$state, pattern = '/', replacement = ''))
matched_row_indices <- match(PHYLOGENY$tip.label, data$binominal)
stopifnot(all(data$binominal[matched_row_indices] == PHYLOGENY$tip.label))
data <- data[matched_row_indices, ] # reorder the dataset to match the species order in the phylogeny
stopifnot(all(data$binominal == PHYLOGENY$tip.label))
stopifnot(length(unique(data$binominal)) == length(data$binominal))

In [4]:
# FOR CONVENIRNCE
RD <- setNames(object = data$RD, nm = data$binominal)
SRL <- setNames(object = data$SRL, nm = data$binominal)
STATES <- setNames(object = data$myco, nm = data$binominal)

In [5]:
table(STATES) # welp

STATES
   AM AMEcM  AMNM   EcM   ErM    NM 
  300    15     8    65     3     4 

### ___ACE of discrete categorical traits using `ape`, `corHMM`, `phytools` & `diversitree`___
------------------------------------------------------

In [5]:
ape::is.ultrametric(PHYLOGENY) # diversitree requires the phylogeny to be ultrametric

[1] TRUE

In [6]:
# diversitree::make.mkn requires the states to be encodes as numbers
STATES_TO_NUMERIC <- setNames(seq_along(unique(STATES)), unique(STATES)) # numeric encoding of the discrete character states
STATES_NUMERIC <- setNames(unname(STATES_TO_NUMERIC[unname(STATES)]), names(STATES)) # named vector for states with numeric encodings

In [7]:
# diversitree::make.mk2() can only be used with binary state discrete characters
fnlikelihood <- diversitree::make.mkn(tree = PHYLOGENY, states = STATES_NUMERIC, k = length(unique(STATES)))

In [8]:
fnlikelihood

Mk(n) likelihood function:
  * Parameter vector takes 30 elements:
     - q12, q13, q14, q15, q16, q21, q23, q24, q25, q26, q31, q32, q34,
       q35, q36, q41, q42, q43, q45, q46, q51, q52, q53, q54, q56, q61,
       q62, q63, q64, q65
  * Function takes arguments (with defaults)
     - pars: Parameter vector
     - root [ROOT.OBS]: Type of root treatment
     - root.p [NULL]: Vector of root state probabilities
     - intermediates [FALSE]: Also return intermediate values?
  * Phylogeny with 395 tips and 394 nodes
     - Taxa: Anaphalis_aureopunctata, Anaphalis_hancockii, ...
  * References:
     - Pagel (1994)
     - Lewis (2001)
R definition:
function (pars, root = ROOT.OBS, root.p = NULL, intermediates = FALSE)

In [40]:
# pars argument of the likelihood function
# for make.mkn, a vector of k(k-1) parameters, in the order q12,q13,...q1k, q21,q23,...,q2k,...qk(k-1),
# corresponding to the off-diagonal elements of the Q matrix in row order. The order of parameters can be seen by running argnames(f)

# in our case k is 6, so we need (6x(6-1)) 30 values???

In [10]:
tm <- Sys.time()
# this is an ARD model
dtmod <- diversitree::find.mle(func = fnlikelihood, x.init = rep(x = 1, times = 30), # x.init and any extra args will get passed to the likelihood function
                      root = diversitree::ROOT.FLAT)
# needs a bit more work to run properly :(
tm <- Sys.time() - tm

Warning message in mle.search(func2, x.init, control, lower, upper):
"Convergence problems in find.mle (subplex): number of function evaluations exceeds 'maxit'"


In [6]:
#----------------------------------------------------------------------------------------------------------------------------------------------------------
# choosing the ARD model for mycorrhizal state evolution as it has been demonstrated that different transitions happen at different rates
# and some transitions are prectically irreversible compared to others
#----------------------------------------------------------------------------------------------------------------------------------------------------------

tm <- Sys.time()

# there are multiple ways to do ACE of discrete characters 
# ace_mystates_rr <- phytools::rerootingMethod(tree = PHYLOGENY, x = STATES, model = "ARD") - phytools::rerootingMethod method should not be used with non-symmetrical rate models of evolution
# read more here - https://blog.phytools.org/2023/06/decommissioning-rerootingmethod-and.html
ace_mystates_hmm <- corHMM::corHMM(phy = PHYLOGENY, data = data[, c("binominal", "myco")], model = "ARD", node.states = "marginal", rate.cat = 1)
ace_mystates_ape <- ape::ace(phy = PHYLOGENY, x = STATES, type = "discrete", method = "ML", model = "ARD", marginal = FALSE) # ape::ace used a non traditional way to compute marginal ACEs, see its documentation for more details
# setting marginal = FALSE actually returns the marginal ACEs, see the documentation of ape::ace for more details
ace_mystates_ancr <- phytools::ancr(phytools::fitMk(tree = PHYLOGENY, x = STATES, model = "ARD"))

tm <- Sys.time() - tm

# ace_mystates_dvt <- diversitree::asr.marginal()

You specified 'fixed.nodes=FALSE' but included a phy object with node labels. These node labels have been removed.


Warning message in corHMM::corHMM(phy = PHYLOGENY, data = data[, c("binominal", :
"Branch lengths of 0 detected. Adding 1e-5 to these branches."


State distribution in data:
States:	1	2	3	4	5	6	
Counts:	300	15	8	65	3	4	
Beginning thorough optimization search -- performing 0 random restarts 
Finished. Inferring ancestral states using marginal reconstruction. 


Warning message in sqrt(diag(solve(h))):
"NaNs produced"


In [7]:
tm

Time difference of 12.70665 mins

In [8]:
save(ace_mystates_hmm, ace_mystates_ape, ace_mystates_ancr, file = "../rdata/statesARD.RData")

In [59]:
ace_mystates_ape$lik.anc # ACEs from ape::ace

,AM,AMEcM,AMNM,EcM,ErM,NM
,0.1780089,0.1676518788,0.1678401104,1.578816e-01,1.634363e-01,1.651812e-01
Spermatophyta,0.6310923,0.0084811117,0.0046860970,3.366761e-01,1.046941e-02,8.594939e-03
Mesangiospermae,0.9977983,0.0008959747,0.0007030378,4.115483e-05,1.866265e-04,3.749249e-04
mrcaott2ott121,0.9976928,0.0009168643,0.0008248789,5.344119e-06,1.821930e-04,3.778822e-04
eudicotyledons,0.9955824,0.0018530047,0.0015131992,1.288869e-04,3.336534e-04,5.888886e-04
mrcaott2ott969,0.9940670,0.0023076272,0.0023267179,7.443955e-05,4.406896e-04,7.835524e-04
Pentapetalae,0.9924739,0.0039979010,0.0019058349,5.098264e-05,5.261451e-04,1.045281e-03
mrcaott248ott19688,0.9834832,0.0065906228,0.0053868733,6.934968e-04,1.511159e-03,2.334679e-03
mrcaott248ott557,0.9799111,0.0082624305,0.0065630318,3.467028e-04,1.854204e-03,3.062496e-03
mrcaott248ott27233,0.9797556,0.0108912152,0.0042882178,4.049869e-04,1.664066e-03,2.995946e-03


In [9]:
ace_mystates_ancr$ace # ACEs from phytools::ancr

,AM,AMEcM,AMNM,EcM,ErM,NM
396,0.0922244,5.397558e-05,4.223003e-01,6.549021e-04,0,4.847664e-01
397,0.5235107,7.513800e-05,2.002798e-01,6.545492e-04,0,2.754798e-01
398,0.9998722,3.853322e-14,7.447998e-05,1.675982e-08,0,5.331681e-05
399,0.9998850,1.748693e-12,6.994310e-05,6.788944e-09,0,4.503764e-05
400,0.9997399,1.083204e-11,2.386851e-04,2.615169e-08,0,2.134789e-05
401,0.9998144,1.764122e-10,1.683307e-04,6.426645e-08,0,1.716133e-05
402,0.9999619,1.840187e-13,2.902678e-05,4.559135e-09,0,9.086524e-06
403,0.9992811,1.071614e-10,5.613456e-04,5.538077e-08,0,1.574814e-04
404,0.9994928,1.314541e-09,3.961422e-04,2.140568e-07,0,1.108851e-04
405,0.9999542,2.030403e-11,3.640801e-05,4.438356e-08,0,9.361716e-06


In [11]:
ace_mystates_hmm$states # ACEs from corHMM

"(1,R1)","(2,R1)","(3,R1)","(4,R1)","(5,R1)","(6,R1)"
0.7384633,1.040288e-04,9.949760e-02,6.260653e-03,6.030534e-04,1.550713e-01
0.8501157,7.903288e-04,7.470484e-02,5.612279e-03,2.847190e-05,6.874840e-02
0.9991916,9.818027e-13,8.082208e-04,7.337873e-08,1.788241e-15,1.214591e-07
0.9992163,2.685532e-11,7.836026e-04,3.027913e-08,2.436969e-14,8.614221e-08
0.9989217,2.534694e-10,1.078005e-03,5.338445e-08,3.126602e-12,2.003021e-07
0.9990869,3.113141e-09,9.128237e-04,1.242300e-07,2.405557e-12,1.708078e-07
0.9995788,3.012160e-12,4.211973e-04,8.369504e-09,7.965788e-15,9.662081e-09
0.9973595,1.671280e-09,2.638908e-03,9.831632e-08,1.065142e-09,1.522477e-06
0.9979293,1.450902e-08,2.069273e-03,3.650733e-07,5.633531e-11,1.053377e-06
0.9993963,3.182802e-10,6.035254e-04,8.145866e-08,1.258612e-13,5.588549e-08


In [12]:
# the output of corHMM lacks explicit column names
# outputs of ape and corHMM lack explicit row names (in node numbers)

In [33]:
# colnames(ace_mystates_ape$lik.anc)
# MARGIN = 1 is used for row wise apply
state_preds_ape <- colnames(ace_mystates_ape$lik.anc)[apply(ace_mystates_ape$lik.anc, MARGIN = 1, FUN = which.max)]

In [34]:
ace_mystates_hmm


Fit
      -lnL      AIC     AICc Rate.cat ntax
 -127.3412 314.6825 319.7924        1  395

Legend
      1       2       3       4       5       6 
   "AM" "AMEcM"  "AMNM"   "EcM"   "ErM"    "NM" 

Rates
            (1,R1)       (2,R1)       (3,R1)      (4,R1)       (5,R1)
(1,R1)          NA 0.0003220296 0.0009737899 0.000201527 6.729156e-05
(2,R1) 0.000000001           NA 0.0000000010 0.017445718 1.000000e-09
(3,R1) 0.016863141 0.0000000010           NA 0.000000001 1.000000e-09
(4,R1) 0.001299564 0.0000000010 0.0000000010          NA 1.000000e-09
(5,R1) 0.000000001 0.0000000010 0.0000000010 0.000000001           NA
(6,R1) 0.000000001 0.0000000010 0.0071694536 0.000000001 1.000000e-09
             (6,R1)
(1,R1) 1.000000e-09
(2,R1) 1.000000e-09
(3,R1) 1.027515e-02
(4,R1) 1.000039e-09
(5,R1) 9.735390e-04
(6,R1)           NA

Arrived at a reliable solution 

In [35]:
# based on the legend of the corHMM model output
STATES_LEGEND_CORHMM <- c("AM", "AMEcM", "AMNM", "EcM", "ErM", "NM")
STATES_LEGEND_CORHMM

[1] "AM"    "AMEcM" "AMNM"  "EcM"   "ErM"   "NM"

In [36]:
state_preds_hmm <- STATES_LEGEND_CORHMM[apply(ace_mystates_hmm$states, MARGIN = 1, FUN = which.max)]

In [37]:
# colnames(ace_mystates_ancr$ace)
# only the output of phytools::ancr has explicit node numbers as row names!!
state_preds_phy <- colnames(ace_mystates_ancr$ace)[apply(ace_mystates_ancr$ace, MARGIN = 1, FUN = which.max)]

In [39]:
mean(state_preds_ape == state_preds_hmm)

[1] 0.9720812

In [40]:
mean(state_preds_ape == state_preds_phy) # the lowest

[1] 0.9492386

In [41]:
mean(state_preds_phy == state_preds_hmm) # the highest

[1] 0.9771574

In [44]:
# let's stick to the reconstructions of corHMM
internode_states <- setNames(state_preds_hmm, nm = rownames(ace_mystates_ancr$ace))

In [53]:
all(rownames(ace_mystates_ancr$ace) == 396:789)

[1] TRUE

In [75]:
stopifnot(all(PHYLOGENY$tip.label == names(STATES)))
all_states <- c(setNames(unname(STATES), nm = as.character(1:395)), # tips
                                    internode_states) # internodes
stopifnot(length(all_states) == PHYLOGENY$Nnode + length(PHYLOGENY$tip.label))

In [ ]:
ace_mystates_dvt <- diversitree::

In [76]:
#--------------------------------
# ACE OF CONTINUOUS TRAITS
#--------------------------------

ace_srl_fastanc <- phytools::fastAnc(tree = PHYLOGENY, x = SRL) # this is a Maximim Likelihood (ML) based method - Brownian motion models which essentially is a Markov process on a continuous scale
# https://eeob-macroevolution.github.io/Practicals/Ancestral_State_Estimation/AncStateEstimation_Tutorial.html
# the link above provides detailed info on ACE, and also recommends incorporating fossil data when available to make the ACE more rigorous
# do we have fossil root trait data???????

# Liam J. Revell's website says that phytools::fastAnc also uses Felsenstein's re-rooting algorithm for ACE of continuous characters - http://www.phytools.org/Cordoba2017/ex/7/Anc-states-continuous.html
# PCMIR (page 224) recommends fitting phytools::fastAnc with vars=TRUE, CI=TRUE as ACE happens with lots of uncertainty, so it's important to know the CIs of each reconstruction

In [84]:
all_srl <- c(setNames(SRL, nm = as.character(1:395)), # tip SRL values
                      ace_srl_fastanc) # reconstructed SRL values
stopifnot(length(all_srl) == PHYLOGENY$Nnode + length(PHYLOGENY$tip.label))

In [92]:
nodedata <- data.frame(SRL = all_srl, STATE = all_states) # includes the extant and reconstructed clade trait data
head(nodedata)

,SRL,STATE
,<dbl>,<chr>
1,479.1550,AMNM
2,319.8579,AM
3,202.8837,AM
4,219.2008,AM
5,297.1100,AM
6,204.1529,AM


In [ ]:
# the plan here is to find out the (2) nodes at each branch of the phylogeny and capture and store the changes in SRL and mycorrhizal states
# corresponding to the branches
# we can start this with the internodes as the tips can only be at the descendent end of a branch, never at an ancestor end

In [91]:
head(PHYLOGENY$edge) # the edge attribute of the phylo object contains all the edge details, where each row corresponds to a branch in the phylogeny

396,397
397,398
398,399
399,400
400,401
401,402


In [123]:
nodedata[345, "SRL"] - nodedata[231, "SRL"]

[1] -65.70696

In [131]:
# figure out all the discrete state transitions included in our reconstruction

transitions <- list()
changes <- vector(length = nrow(PHYLOGENY$edge), mode = "double")

for (i in 1:nrow(PHYLOGENY$edge)) {
    # PHYLOGENY$edge[i, ] returns a tuple of node numbers defining a branch (from, to)
    from <- PHYLOGENY$edge[i, ][1]
    to <- PHYLOGENY$edge[i, ][1]

    # corresponding state transition
    transitions[[i]] <- c(nodedata[from, "STATE"], nodedata[to, "STATE"])

    # corresponding continuous trait change
    changes[i] <- (nodedata[from, "SRL"] - nodedata[to, "SRL"]) # SRL of ancestor - SRL of the descendent
    print(paste0(nodedata[from, "SRL"], ' ', nodedata[to, "SRL"])) # that's fucked up :/
}

[1] "87.1887153768475 87.1887153768475"
[1] "74.3876076639289 74.3876076639289"
[1] "86.4443514259488 86.4443514259488"
[1] "86.8177302379011 86.8177302379011"
[1] "90.6010055805565 90.6010055805565"
[1] "90.7782055189518 90.7782055189518"
[1] "94.5855739567672 94.5855739567672"
[1] "96.9510153424221 96.9510153424221"
[1] "98.664126564583 98.664126564583"
[1] "104.677020630068 104.677020630068"
[1] "107.671044067949 107.671044067949"
[1] "112.587736400722 112.587736400722"
[1] "115.830567394651 115.830567394651"
[1] "129.237740416893 129.237740416893"
[1] "222.481007356438 222.481007356438"
[1] "232.332123408069 232.332123408069"
[1] "247.855763416357 247.855763416357"
[1] "257.153239504369 257.153239504369"
[1] "394.527379966816 394.527379966816"
[1] "394.527379966816 394.527379966816"
[1] "257.153239504369 257.153239504369"
[1] "236.116948931954 236.116948931954"
[1] "240.248685710358 240.248685710358"
[1] "237.272275554028 237.272275554028"
[1] "237.272275554028 237.272275554028"
[1

In [127]:
changes

[1] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 [38] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 [75] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[112] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[149] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[186] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[223] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[260] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[297] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[334] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[371] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[408] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[445] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[482] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[519] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[556] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[593] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[630] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[667] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[704] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[741] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[778] 0 0 0 0 0 0 0 0 0 0 0